In [64]:
from pathlib import Path

import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

In [66]:
print("Fetching Online Retail dataset from the UCI ...")
online_retail = fetch_ucirepo(id=352)

Fetching Online Retail dataset from the UCI ...


In [67]:
X = online_retail.data.features
ids = online_retail.data.ids
df = ids.join(X)

print(f"Raw shape: {df.shape}")
print("Missing values per column:\n", df.isnull().sum(), sep="")

Raw shape: (541909, 8)
Missing values per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Cleaning

In [69]:
## Drop rows without CustomerID
df = df.dropna(subset=["CustomerID"]).copy()

# Fill missing product descriptions
df["Description"] = df["Description"].fillna("Unknown Product")

# Remove duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows removed: {dup_count}")
df = df.drop_duplicates().reset_index(drop=True)

# Drop non-product / administrative StockCodes
#     (postage, manual adjustments, bank charges, Amazon fees, bad debt, etc.)
bad_codes = {"POST", "D", "C2", "M", "BANK CHARGES", "AMAZONFEE",
             "DOT", "CRUK", "PADS", "B"}
df = df[~df["StockCode"].astype(str).str.upper().isin(bad_codes)]

# Cancellations: InvoiceNo starting with 'C' = cancelled order.
#     Keep them in a separate frame for return-behaviour features, then
#     drop from the main transactions frame.
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
cancellations = df[df["InvoiceNo"].str.startswith("C")].copy()
df = df[~df["InvoiceNo"].str.startswith("C")]

# Keep only strictly positive quantities and prices.
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# Types
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)

# Derived row-level columns
df["Amount"] = df["Quantity"] * df["UnitPrice"]           # line revenue
df["Date"]   = df["InvoiceDate"].dt.date
df["Time"]   = df["InvoiceDate"].dt.time
df["Hour"]   = df["InvoiceDate"].dt.hour
df["DOW"]    = df["InvoiceDate"].dt.day_name()
df["Month"]  = df["InvoiceDate"].dt.month


print(f"Clean transaction shape: {df.shape}")
print(f"Date range: {df['InvoiceDate'].min()}  ->  {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique invoices : {df['InvoiceNo'].nunique()}")
print(f"Countries       : {df['Country'].nunique()}")


Duplicate rows removed: 5225
Clean transaction shape: (391150, 14)
Date range: 2010-12-01 08:26:00  ->  2011-12-09 12:50:00
Unique customers: 4334
Unique invoices : 18402
Countries       : 37


## Feature Engineering

In [70]:
## Reference data = day after last transaction, the standard RFM conversion
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# RFM core
rfm = df.groupby("CustomerID").agg(
    Recency   = ("InvoiceDate", lambda s: (snapshot_date - s.max()).days),
    Frequency = ("InvoiceNo",   "nunique"),     # #distinct invoices
    Monetary  = ("Amount",      "sum"),
)

# Extra behavioural features (useful for segmentation beyond pure RFM)
extras = df.groupby("CustomerID").agg(
    TotalQuantity      = ("Quantity",   "sum"),
    UniqueProducts     = ("StockCode",  "nunique"),
    AvgUnitPrice       = ("UnitPrice",  "mean"),
    AvgBasketValue     = ("Amount",     "mean"), 
    FirstPurchase      = ("InvoiceDate", "min"),
    LastPurchase       = ("InvoiceDate", "max"),
)

# Average order value (per invoice) and items per order
order_totals = df.groupby(["CustomerID", "InvoiceNo"]).agg(
    order_value = ("Amount",   "sum"),
    order_items = ("Quantity", "sum"),
).reset_index()

per_order = order_totals.groupby("CustomerID").agg(
    AvgOrderValue = ("order_value", "mean"),
    AvgOrderItems = ("order_items", "mean"),
)

# Customer tenure and inter-purchase rhythm
extras["TenureDays"] = (extras["LastPurchase"] - extras["FirstPurchase"]).dt.days

# Return behaviour — number of cancelled invoices per customer.
cancellations["CustomerID"] = pd.to_numeric(
    cancellations["CustomerID"], errors="coerce"
)
returns = (
    cancellations.dropna(subset=["CustomerID"])
    .assign(CustomerID=lambda d: d["CustomerID"].astype(int))
    .groupby("CustomerID")
    .agg(ReturnInvoices=("InvoiceNo", "nunique"))
)

# Dominant country per customer (mode)
country = (
    df.groupby("CustomerID")["Country"]
    .agg(lambda s: s.mode().iat[0])
    .rename("Country")
)

# Assemble
customer_features = (
    rfm
    .join(extras)
    .join(per_order)
    .join(returns)
    .join(country)
)
customer_features["ReturnInvoices"] = customer_features["ReturnInvoices"].fillna(0).astype(int)
customer_features["ReturnRate"] = (
    customer_features["ReturnInvoices"] /
    (customer_features["Frequency"] + customer_features["ReturnInvoices"])
)

# Log-transformed variants — RFM is extremely right-skewed and clustering
#     algorithms (k-means, GMM, hierarchical w/ Euclidean) behave much better
#     on log-scaled features. We export both; pick in R.
for col in ["Recency", "Frequency", "Monetary",
            "TotalQuantity", "UniqueProducts", "AvgOrderValue"]:
    customer_features[f"log_{col}"] = np.log1p(customer_features[col])

customer_features = customer_features.reset_index()

In [80]:
# Save
# ---------------------------------------------------------------------------
import os
out_dir = Path.cwd().parent
os.makedirs(out_dir / "data", exist_ok=True)
df.to_csv(out_dir/ "data" / "online_retail_clean.csv", index=False)
customer_features.to_csv(out_dir / "data" / "customer_features.csv", index=False)

print("\nSaved:")
print(" ", out_dir / "data" / "online_retail_clean.csv", df.shape)
print(" ", out_dir / "data" / "customer_features.csv", customer_features.shape)

print("\nCustomer feature preview:")
print(customer_features.head())
print("\nSummary stats:")
print(customer_features[["Recency", "Frequency", "Monetary",
                         "AvgOrderValue", "TenureDays", "ReturnRate"]]
      .describe().round(2))



Saved:
  /mnt/c/Users/anish/CSP_571_FinalProj/data/online_retail_clean.csv (391150, 14)
  /mnt/c/Users/anish/CSP_571_FinalProj/data/customer_features.csv (4334, 22)

Customer feature preview:
   CustomerID  Recency  Frequency  Monetary  TotalQuantity  UniqueProducts  \
0       12346      326          1  77183.60          74215               1   
1       12347        2          7   4310.00           2458             103   
2       12348       75          4   1437.24           2332              21   
3       12349       19          1   1457.55            630              72   
4       12350      310          1    294.40            196              16   

   AvgUnitPrice  AvgBasketValue       FirstPurchase        LastPurchase  ...  \
0      1.040000    77183.600000 2011-01-18 10:01:00 2011-01-18 10:01:00  ...   
1      2.644011       23.681319 2010-12-07 14:57:00 2011-12-07 15:52:00  ...   
2      0.692963       53.231111 2010-12-16 19:09:00 2011-09-25 13:13:00  ...   
3      4.237500   